In [2]:
#Test the connection
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = quote_plus(os.getenv("DB_PASSWORD"))  # URL-encode special characters

connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
print("Connecting with user:", DB_USER)  # debug check

engine = create_engine(connection_string)

query = """
SELECT id, title, price, location, constituency, address, 
       bedrooms, bathrooms, floor_size, excerpt
FROM property_listings
WHERE price IS NOT NULL
"""

df = pd.read_sql(query, engine)
print(f"Loaded {len(df)} rows from database")
df.head()

Connecting with user: ml_user_2.krdnjiphdvdfeqqrxvji
Loaded 1740 rows from database


,id,title,price,location,constituency,address,bedrooms,bathrooms,floor_size,excerpt
0,1,4 Bedroom Apartment / Flat,27000000.0,Westlands,Westlands,"Westlands, Westlands, Nairobi",4.0,4.0,211.0,Duplex Apartments & Penthouses for Sale – West...
1,2,1 Bedroom Apartment / Flat,9500000.0,Westlands,Westlands,"Westlands, Westlands, Nairobi",1.0,1.0,58.0,Welcome to an iconic new development located a...
2,3,1 Bedroom Apartment / Flat,10100000.0,Westlands,Westlands,"Westlands, Westlands, Nairobi",1.0,1.0,90.0,Elegantly designed 1 & 2 bedroom luxury units ...
3,4,1 Bedroom Apartment / Flat,6100000.0,Kilimani,Langata,"Kilimani, Kilimani, Nairobi",1.0,1.0,55.0,"Book your unit with 100,000 today!\r\nLocated ..."
4,5,4 Bedroom Apartment / Flat,33500000.0,Westlands,Westlands,"Westlands, Westlands, Nairobi",4.0,4.0,213.0,Discover the ultimate convenience and lifestyl...


In [3]:
#Cell 2 -Save snapshot +inspect
import os

os.makedirs('../data', exist_ok=True)
df.to_csv('../data/property_listings_db.csv', index=False)
print(f"Saved snapshot: {len(df)} rows")

print(df.isnull().sum())
print(f"\nbedrooms range: {df['bedrooms'].min()} - {df['bedrooms'].max()}")
print(f"price range: {df['price'].min():,.0f} - {df['price'].max():,.0f}")

Saved snapshot: 1740 rows
id                0
title             0
price             0
location          0
constituency      0
address           0
bedrooms         53
bathrooms        45
floor_size      180
excerpt           0
dtype: int64

bedrooms range: 1.0 - 11.0
price range: 120,000 - 600,000,000


In [5]:
#Cell 3 — Clean & feature engineer
import re
from sklearn.preprocessing import LabelEncoder

# Dedup
df = df.drop_duplicates(subset=['title', 'price', 'address', 'floor_size'])
print(f"After dedup: {df.shape}")

# Property type from title
df['property_type'] = df['title'].str.extract(
    r'(Apartment|House|Villa|Townhouse|Studio|Penthouse)', flags=re.IGNORECASE
)
df['property_type'] = df['property_type'].fillna('Other').str.title()

# Drop rows missing critical fields
df = df.dropna(subset=['price', 'bedrooms', 'bathrooms', 'floor_size', 'constituency'])
print(f"After dropping nulls: {df.shape}")

print(df['property_type'].value_counts())
print(df['constituency'].value_counts())

After dedup: (632, 11)
After dropping nulls: (632, 11)
property_type
Apartment    492
House         71
Townhouse     69
Name: count, dtype: int64
constituency
Westlands        376
Langata          229
Mavoko            12
Unknown           11
Roysambu           3
Embakasi East      1
Name: count, dtype: int64


In [6]:
# Cell 4 -  One-Hot encode constituency
const_dummies = pd.get_dummies(df['constituency'], prefix='const')
df = pd.concat([df, const_dummies], axis=1)
const_cols = [c for c in df.columns if c.startswith('const_')]

# Label encode property type
le_type = LabelEncoder()
df['type_encoded'] = le_type.fit_transform(df['property_type'])

feature_cols = ['bedrooms', 'bathrooms', 'floor_size', 'type_encoded'] + const_cols
print(f"Total features: {len(feature_cols)}")
print(feature_cols)

Total features: 10
['bedrooms', 'bathrooms', 'floor_size', 'type_encoded', 'const_Embakasi East', 'const_Langata', 'const_Mavoko', 'const_Roysambu', 'const_Unknown', 'const_Westlands']


In [7]:
# Cell 5 — Build training set + save
import pickle

target_col = 'price'
train_df = df[feature_cols + [target_col]].dropna()
print(f"Training dataset shape: {train_df.shape}")

train_df.to_csv('../data/train_ready.csv', index=False)

os.makedirs('../models', exist_ok=True)
with open('../models/encoders.pkl', 'wb') as f:
    pickle.dump({'property_type': le_type}, f)

print("✅ train_ready.csv saved")

Training dataset shape: (632, 11)
✅ train_ready.csv saved


In [10]:
#The initial data with 1700~ rows was not clean and we need to check for the outlier  then we check

# Check for outliers
print("Price distribution:")
print(train_df['price'].describe())

print("\nTop 10 highest prices:")
print(df.nlargest(10, 'price')[['title', 'price', 'floor_size', 'bedrooms', 'constituency']])

print("\nBottom 10 lowest prices:")
print(df.nsmallest(10, 'price')[['title', 'price', 'floor_size', 'bedrooms', 'constituency']])

Price distribution:
count    6.320000e+02
mean     3.787573e+07
std      5.462841e+07
min      1.200000e+05
25%      9.450000e+06
50%      1.515000e+07
75%      3.350000e+07
max      5.200000e+08
Name: price, dtype: float64

Top 10 highest prices:
                    title        price  floor_size  bedrooms constituency
1419      6 Bedroom House  520000000.0       650.0       6.0    Westlands
832       5 Bedroom House  350000000.0       650.0       5.0    Westlands
676       3 Bedroom House  270000000.0       180.0       3.0      Langata
1137      3 Bedroom House  270000000.0       300.0       3.0      Langata
838   5 Bedroom Townhouse  259000000.0       300.0       5.0    Westlands
839       5 Bedroom House  259000000.0       300.0       5.0    Westlands
735       5 Bedroom House  250000000.0       400.0       5.0    Westlands
423   5 Bedroom Townhouse  220000000.0       900.0       5.0      Langata
424       5 Bedroom House  220000000.0       900.0       5.0      Langata
762       5 

In [11]:
#Cell 6 
# Fix floor_size data entry errors (values >2000 are likely missing a decimal point) -- Run this before cell 6 
train_df = train_df[train_df['floor_size'] < 2000]
print(f"After floor_size fix: {len(train_df)} rows")

# Remove price outliers — use percentile-based filtering
lower = train_df['price'].quantile(0.02)
upper = train_df['price'].quantile(0.98)
print(f"Keeping prices between KSh {lower:,.0f} and KSh {upper:,.0f}")

train_df = train_df[(train_df['price'] >= lower) & (train_df['price'] <= upper)]
print(f"After price outlier removal: {len(train_df)} rows")

print(train_df['price'].describe())

After floor_size fix: 625 rows
Keeping prices between KSh 5,048,000 and KSh 195,000,000
After price outlier removal: 602 rows
count    6.020000e+02
mean     3.445967e+07
std      4.403502e+07
min      5.100000e+06
25%      9.610000e+06
50%      1.520000e+07
75%      3.175000e+07
max      1.950000e+08
Name: price, dtype: float64


In [12]:
#Run Cell 6 before running this to remove dedup,outlier and fix floor_size data error
# Cell 7 — Train/test split + models
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

X = train_df.drop(columns=['price'])
y = train_df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

def report(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"=== {name} ===")
    print(f"RMSE: KSh {rmse:,.0f}")
    print(f"MAE:  KSh {mae:,.0f}")
    print(f"R²:   {r2:.4f}\n")
    return rmse, mae, r2

report("Linear Regression", y_test, y_pred_lr)
report("XGBoost", y_test, y_pred_xgb)

Train: 481 rows, Test: 121 rows
=== Linear Regression ===
RMSE: KSh 23,497,477
MAE:  KSh 14,391,170
R²:   0.7158

=== XGBoost ===
RMSE: KSh 17,919,377
MAE:  KSh 7,874,008
R²:   0.8347



(np.float64(17919376.68898823), 7874008.340909091, 0.8347130475121605)

In [14]:

#Test the model and its perfomance 
import pandas as pd

# Test property — 3 bed, 2 bath, 120m², Westlands apartment
test_property = {
    'bedrooms': 3,
    'bathrooms': 2.0,
    'floor_size': 120,
    'property_type': 'Apartment',
    'constituency': 'Westlands'
}

# Encode property type
type_encoded = le_type.transform([test_property['property_type']])[0]

# One-Hot encode constituency — match training columns exactly
all_constituencies = ['Embakasi East', 'Langata', 'Mavoko', 'Roysambu', 'Unknown', 'Westlands']
const_features = {f"const_{c}": (1 if c == test_property['constituency'] else 0) for c in all_constituencies}

feature_row = pd.DataFrame([{
    'bedrooms': test_property['bedrooms'],
    'bathrooms': test_property['bathrooms'],
    'floor_size': test_property['floor_size'],
    'type_encoded': type_encoded,
    **const_features
}])

predicted_price = xgb.predict(feature_row)[0]
print(f"Predicted price: KSh {predicted_price:,.0f}")
print(f"That's KSh {predicted_price/1e6:.2f}M")

Predicted price: KSh 15,963,006
That's KSh 15.96M


In [16]:
#Final Cell to save model 

import pickle

rmse_xgb = 17919376.68898823
mae_xgb = 7874008.340909091
r2_xgb = 0.8347130475121605

with open('../models/stage1_price_model.pkl', 'wb') as f:
    pickle.dump(xgb, f)

print("✅ Model saved — trained on", len(train_df), "rows after cleaning")
print(f"\n=== Final XGBoost Stage 1 Metrics ===")
print(f"RMSE: KSh {rmse_xgb:,.0f}")
print(f"MAE:  KSh {mae_xgb:,.0f}")
print(f"R²:   {r2_xgb:.4f} — explains {r2_xgb*100:.1f}% of price variance")

✅ Model saved — trained on 602 rows after cleaning

=== Final XGBoost Stage 1 Metrics ===
RMSE: KSh 17,919,377
MAE:  KSh 7,874,008
R²:   0.8347 — explains 83.5% of price variance
